<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 6 — The Unix File System and the Command Line

A word-frequency pipeline built entirely from small shell tools.

## What this notebook does

We will answer one question — **what are the most frequent words in Shakespeare?** — without writing a
program. Instead we will connect a handful of small Unix tools, each doing one thing, into a pipeline.

Along the way we cover the material of this session:

1. Where you are in the file system, and how paths name locations (`pwd`, `ls`, `cd`, `.`, `..`, `~`).
2. Building a workspace (`mkdir`, `cp`, `mv`, `rm`).
3. Looking at files without opening them (`head`, `tail`, `wc`, `grep`).
4. The three standard streams, and redirecting them with `>`, `>>`, `<`, and `2>`.
5. Pipes (`|`), which is where the word-frequency pipeline comes from.
6. Turning the pipeline into an executable script, and the permissions that make it runnable.
7. Running a Python script from the shell, and the `pathlib` / `os` / `sys` equivalents of these commands.

Everything here also works — in fact works *better* — in a real terminal. Open one alongside this
notebook and try the commands there too.

> **Safety principle.** The shell does exactly what you type, immediately. `rm` has no undo and no
> trash can. Read a command before pressing Enter, and check `pwd` before you delete or move anything.

## Running shell commands from a notebook

Two ways:

- `%%sh` as the **first line of a cell** sends the whole cell to a shell.
- `!command` runs a single command inline, and `!` also works inside Python code.

One important catch: **each `%%sh` cell runs in its own fresh shell**. A `cd` in one cell does not
carry over to the next — the next cell starts again in the notebook's directory. That is why every
cell below uses paths relative to this notebook instead of relying on an earlier `cd`.

In [ ]:
%%sh
echo "Hello from $SHELL"
date

## 1. Where am I? The working directory

Every shell session has a **current working directory**. Relative paths are interpreted from it, so
knowing it is the difference between reading the file you meant and getting `No such file or directory`.

- `pwd` — print working directory
- `ls` — list a directory (`-l` long, `-h` human-readable sizes, `-a` including hidden files)

In [ ]:
%%sh
pwd
echo "---"
ls -la

### Paths: absolute and relative

An **absolute path** starts at the root of the tree, `/`, and means the same thing from anywhere:
`/Users/student/ifi8410/06-Unix-Command-Line`.

A **relative path** is interpreted from the working directory and does not start with `/`:
`data/shakespeare.txt`.

Three shorthands appear constantly:

| Notation | Meaning |
|---|---|
| `~` | your home directory |
| `.` | the current directory |
| `..` | the parent of the current directory |

Remember that `cd` inside a `%%sh` cell only lasts for that cell.

In [ ]:
%%sh
echo "Home directory:   $HOME"
echo "Working here:     $(pwd)"
echo "The parent is:    $(cd .. && pwd)"
echo
echo "Sessions in the course repository:"
ls ..

## 2. Building a workspace

`mkdir -p` creates a directory, and with `-p` it creates missing parents and does not complain if the
directory already exists — which makes it safe to re-run.

We will use two directories: `data/` for inputs and `out/` for anything we produce. Keeping inputs and
outputs apart is a habit worth forming: you can delete `out/` and rebuild it without risking your data.

In [ ]:
%%sh
mkdir -p data out
ls -l

### Copying, moving, removing

- `cp source target` — copy (`cp -r` for a whole directory)
- `mv source target` — move, and also **rename**: moving a file to a new name in the same directory
- `rm file` — remove, permanently

`cp` and `mv` overwrite the target without asking. `rm -i` asks before each file; `rm -r` removes a
directory and everything inside it. Combine `-r` with a wildcard only after you have checked what the
wildcard matches — run `ls` on it first.

In [ ]:
%%sh
# A scratch file to practice on, so nothing important is at risk.
echo "first line" > out/scratch.txt
cp out/scratch.txt out/copy.txt
mv out/copy.txt out/renamed.txt
ls -l out
echo "--- after rm ---"
rm out/renamed.txt
ls -l out

## 3. Getting the data

Shakespeare's complete works are on [Project Gutenberg](https://www.gutenberg.org/ebooks/100) as plain
text. `curl -L -o target url` downloads it; `-L` follows redirects, `-o` names the output file.

The `if [ ! -f ... ]` test means the cell only downloads the file when it is not already there, so
re-running the notebook is quick and polite to Gutenberg's servers.

In [ ]:
%%sh
if [ ! -f data/pg100.txt ]; then
  curl -L -s -o data/pg100.txt https://www.gutenberg.org/cache/epub/100/pg100.txt
  echo "downloaded"
else
  echo "already present"
fi
ls -lh data

## 4. Looking at a file without opening it

A 5 MB text file is awkward in an editor and pointless to `print()`. The shell has purpose-built tools:

- `head -n 5 file` — the first lines
- `tail -n 5 file` — the last lines
- `wc file` — count lines, words, and characters (`wc -l` for lines only)
- `less file` — page through it interactively (**in a terminal**, not here: `q` quits, `/` searches)
- `grep pattern file` — print the lines that match

In [ ]:
%%sh
echo "=== first 8 lines ==="
head -n 8 data/pg100.txt
echo
echo "=== lines / words / characters ==="
wc data/pg100.txt

### The text is not all Shakespeare

The file is wrapped in a Project Gutenberg licence header and footer. Those lines are English text too,
and they would land in our word counts. The boundaries are marked explicitly:

In [ ]:
%%sh
grep -n "PROJECT GUTENBERG EBOOK" data/pg100.txt

`sed -n '/START/,/END/p'` prints only the lines **between** two matching patterns — a range, rather than
a single line. We save the result as `data/shakespeare.txt` and work from that from here on.

In [ ]:
%%sh
sed -n '/^\*\*\* START OF THE PROJECT GUTENBERG/,/^\*\*\* END OF THE PROJECT GUTENBERG/p' \
    data/pg100.txt > data/shakespeare.txt

wc -l data/pg100.txt data/shakespeare.txt

## 5. The three standard streams, and redirection

Every Unix program is born connected to three streams:

| Stream | Number | Default |
|---|---|---|
| standard input (`stdin`) | 0 | the keyboard |
| standard output (`stdout`) | 1 | the terminal |
| standard error (`stderr`) | 2 | the terminal |

Redirection re-points them at files:

- `> file` — send stdout to a file, **replacing** its contents
- `>> file` — append to a file instead
- `< file` — read stdin from a file
- `2> file` — send stderr to a file (`2>/dev/null` discards it)

The distinction between stdout and stderr is what lets you save results and error messages separately.

In [ ]:
%%sh
# > creates or overwrites; >> appends
echo "run 1" >  out/log.txt
echo "run 2" >> out/log.txt
cat out/log.txt

echo
# stdout and stderr go to different places
ls data/shakespeare.txt data/does-not-exist.txt > out/found.txt 2> out/errors.txt
echo "found:"; cat out/found.txt
echo "errors:"; cat out/errors.txt

## 6. Pipes: the word-frequency pipeline

A pipe (`|`) connects the stdout of one program to the stdin of the next, with no temporary file in
between. This is the Unix philosophy made concrete: small programs that each do one thing, composed
into something none of them does alone.

We build the pipeline one stage at a time, checking the output at each step — the way you should
develop any pipeline.

**Stage 1: one word per line.** `tr` translates or deletes characters. We test it on a few lines
first — `sed -n '65,80p'` prints lines 65 to 80 — because a pipeline is easier to debug on 16 lines
than on 196,000.
- `tr -d '.,:;?!"()[]'` deletes punctuation.
- `tr 'A-Z' 'a-z'` lowercases, so *The* and *the* count as one word.
- `tr -s ' \t\r' '\n'` turns runs of spaces and tabs into single newlines.

In [ ]:
%%sh
sed -n '65,80p' data/shakespeare.txt \
| tr -d '.,:;?!"()[]' \
| tr 'A-Z' 'a-z' \
| tr -s ' \t\r' '\n' \
| head -n 20

**Stage 2: drop the blank lines.** `grep -v` inverts the match — it prints lines that do *not* match.
The pattern `^[[:space:]]*$` matches a line containing nothing but whitespace.

**Stage 3: count.** `uniq -c` collapses *adjacent* identical lines and prefixes each with a count, which
is why it must be preceded by `sort`. Then `sort -rn` sorts by that count, numerically (`-n`) and in
reverse (`-r`), and `head` keeps the top of the list.

In [ ]:
%%sh
tr -d '.,:;?!"()[]' < data/shakespeare.txt \
| tr 'A-Z' 'a-z' \
| tr -s ' \t\r' '\n' \
| grep -v -e '^[[:space:]]*$' \
| sort \
| uniq -c \
| sort -rn \
| head -n 25

The result is unsurprising — *the*, *and*, *i*, *to* — and that is itself a finding: the most frequent
words in any English text are function words. Interesting analysis usually starts by removing them.

Note the shape of what we just did. No file was written, no program was compiled, and each stage could
be inspected on its own. Six tools, none of which knows anything about Shakespeare.

### Asking narrower questions

The same pipeline answers other questions if you change one stage. `grep -w` matches whole words only,
and `-i` ignores case.

In [ ]:
%%sh
for WORD in love hate murder faith; do
  COUNT=$(grep -o -i -w "$WORD" data/shakespeare.txt | wc -l)
  echo "$WORD: $COUNT"
done

### Saving the result

A pipeline ends like any other command, so `>` saves its output. We keep the top 1000 words in `out/`.

In [ ]:
%%sh
tr -d '.,:;?!"()[]' < data/shakespeare.txt \
| tr 'A-Z' 'a-z' \
| tr -s ' \t\r' '\n' \
| grep -v -e '^[[:space:]]*$' \
| sort \
| uniq -c \
| sort -rn \
| head -n 1000 > out/wordfreq.txt

wc -l out/wordfreq.txt
head -n 5 out/wordfreq.txt

## 7. From pipeline to script, and the permissions that run it

We have now typed that pipeline three times. A command you repeat belongs in a **script**.

Three steps:

1. Put the commands in a text file, e.g. `wordfreq.sh`.
2. Make the first line a **hash-bang**: `#!/bin/bash` tells the kernel which interpreter to use.
3. Make the file executable with `chmod +x wordfreq.sh`.

The script below reads from stdin and writes to stdout — no filenames inside it — so it works with any
input, exactly like the built-in tools. That is what makes a script composable.

(The `cat > file << 'EOF' ... EOF` form below is a **heredoc**: everything up to the `EOF` marker
becomes the file's contents. Writing the file in an editor works just as well.)

In [ ]:
%%sh
cat > wordfreq.sh << 'EOF'
#!/bin/bash
# Read text on stdin, write "count word" lines on stdout, most frequent first.
tr -d '.,:;?!"()[]' \
| tr 'A-Z' 'a-z' \
| tr -s ' \t\r' '\n' \
| grep -v -e '^[[:space:]]*$' \
| sort \
| uniq -c \
| sort -rn
EOF

ls -l wordfreq.sh

Look at the permissions in that listing: something like `-rw-r--r--`. Unix records permissions for three
classes of user, in three groups of three characters:

```
-  rw-      r--      r--
   owner    group    others
```

- `r` read, `w` write, `x` execute — a `-` means the permission is absent.
- For a **directory**, `x` means "may enter it"; without it you cannot `cd` in even if you can read it.

`chmod` changes them. `chmod +x file` adds execute permission; the symbolic form `chmod u+x` adds it for
the owner only, and the numeric form `chmod 755` sets `rwxr-xr-x` in one go.

In [ ]:
%%sh
chmod +x wordfreq.sh
ls -l wordfreq.sh
echo
# ./ says "the file in this directory" — the shell only searches $PATH otherwise.
./wordfreq.sh < data/shakespeare.txt | head -n 10

The script is now a tool like any other, so it composes with pipes:

In [ ]:
%%sh
# The 10 most frequent words in Hamlet alone: the lines from its title
# up to the title of the play that follows it.
sed -n '/THE TRAGEDY OF HAMLET/,/THE FIRST PART OF KING HENRY THE FOURTH/p' data/shakespeare.txt \
| ./wordfreq.sh \
| head -n 10

## 8. Running Python from the shell

A Python script is run the same way: `python3 script.py argument`. The shell hands it the words on the
command line, and Python receives them in `sys.argv`:

- `sys.argv[0]` is the script's own name,
- `sys.argv[1]`, `sys.argv[2]`, … are the arguments.

The script below takes the input file and the number of words to show. It also reads a *stop word* from
nowhere but its arguments — all its inputs are explicit, which is what makes it reproducible.

In [ ]:
%%sh
cat > wordfreq.py << 'EOF'
#!/usr/bin/env python3
"""Count word frequencies in a text file.

Usage: python3 wordfreq.py INPUT_FILE [TOP_N]
"""
import sys
from collections import Counter
from pathlib import Path

PUNCTUATION = '.,:;?!"()[]'


def tokenize(text):
    """Split text into lowercase words, stripping punctuation."""
    cleaned = text.translate(str.maketrans("", "", PUNCTUATION))
    return cleaned.lower().split()


def main():
    if len(sys.argv) < 2:
        # Usage errors belong on stderr, not stdout.
        print(f"usage: {sys.argv[0]} INPUT_FILE [TOP_N]", file=sys.stderr)
        return 1

    path = Path(sys.argv[1])
    top_n = int(sys.argv[2]) if len(sys.argv) > 2 else 20

    if not path.is_file():
        print(f"error: no such file: {path}", file=sys.stderr)
        return 1

    text = path.read_text(encoding="utf-8", errors="replace")
    counts = Counter(tokenize(text))

    for word, count in counts.most_common(top_n):
        print(f"{count:>8} {word}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
EOF

python3 wordfreq.py data/shakespeare.txt 10

Two things to notice.

**The exit status.** `main()` returns `0` on success and `1` on failure, and `sys.exit()` passes that to
the shell as the **exit status**. `0` means success; anything else means failure. The shell stores it in
`$?`, and that is how scripts and pipelines decide whether to continue.

**Where the error message went.** The usage message is printed to `sys.stderr`, so it stays on screen
even when stdout is redirected into a file.

In [ ]:
%%sh
python3 wordfreq.py data/shakespeare.txt 5 > out/python_freq.txt
echo "exit status: $?"
cat out/python_freq.txt

echo
# Now with a missing argument: the error appears, but the file stays empty.
python3 wordfreq.py > out/should_be_empty.txt
echo "exit status: $?"
wc -c out/should_be_empty.txt

### Do the two pipelines agree?

The shell pipeline and the Python script split words slightly differently — Python's `.split()` breaks on
any whitespace, while our `tr` handles spaces, tabs, and carriage returns. Comparing their output is a
good use of `diff`, which prints nothing at all when two files are identical.

In [ ]:
%%sh
./wordfreq.sh < data/shakespeare.txt | head -n 5 | sed 's/^ *//' > out/shell_top5.txt
python3 wordfreq.py data/shakespeare.txt 5 | sed 's/^ *//' > out/python_top5.txt

echo "shell:";  cat out/shell_top5.txt
echo "python:"; cat out/python_top5.txt
echo "--- diff (no output means identical) ---"
diff out/shell_top5.txt out/python_top5.txt && echo "identical" 

## 9. The same operations from Python

The shell commands are not magic; they are ordinary programs calling the operating system. Python calls
the same operating system, so every command in this notebook has an equivalent in `pathlib`, `os`, and
`shutil`. Use whichever fits: the shell for interactive work and quick pipelines, Python when the logic
belongs in a program.

| Shell | Python |
|---|---|
| `pwd` | `Path.cwd()` |
| `ls` | `Path(".").iterdir()` |
| `cd` | `os.chdir()` |
| `mkdir -p` | `Path(...).mkdir(parents=True, exist_ok=True)` |
| `cp` | `shutil.copy()` |
| `mv` | `shutil.move()` or `Path.rename()` |
| `rm` | `Path.unlink()` |
| `~` | `Path.home()` |
| `wc -l` | `sum(1 for _ in open(path))` |

Note that `Path` objects join with `/`, which reads naturally and works on every platform — much better
than gluing strings together.

In [ ]:
from pathlib import Path

here = Path.cwd()
print("working directory:", here)
print("home directory:   ", Path.home())

data_file = here / "data" / "shakespeare.txt"
print("\nfile:      ", data_file.name)
print("parent:    ", data_file.parent)
print("suffix:    ", data_file.suffix)
print("exists:    ", data_file.exists())
print("size (MB): ", round(data_file.stat().st_size / 1_000_000, 1))

print("\ntext files under out/:")
for p in sorted(Path("out").glob("*.txt")):
    print(f"  {p}  ({p.stat().st_size} bytes)")

## 10. Your turn

Work in the terminal where you can, and check each step before moving to the next.

1. **Navigate.** From your home directory, reach this session's directory using a relative path only.
   Confirm with `pwd`. Then get back home in one command — three different ways work.
2. **Organize.** Create `out/reports/`, move `out/wordfreq.txt` into it, and list the result. Then make a
   copy named `wordfreq_backup.txt` in the same place.
3. **Inspect.** How many lines of `data/shakespeare.txt` contain the word *king*? (`grep` and `wc`, joined
   by a pipe.) How many contain it capitalized?
4. **Extend the pipeline.** The top of the frequency list is all function words. Write the words *the, and,
   i, to, of, a, my, you, in, that* into `data/stopwords.txt`, one per line, then use
   `grep -v -w -f data/stopwords.txt` inside the pipeline to remove them. What is the most frequent
   *content* word in Shakespeare?
5. **Script it.** Copy `wordfreq.sh` to `wordfreq_clean.sh`, add the stop-word filter, make it executable,
   and run it on `data/shakespeare.txt`.
6. **Compare two texts.** Download another Gutenberg book into `data/`, run your script on both, and use
   `head` and `diff` to compare their top 20 words.
7. **Python.** Extend `wordfreq.py` with a third argument: a stop-word file to ignore. Keep the error
   messages on `sys.stderr` and the results on `sys.stdout`.

## Takeaways

- The file system is one tree rooted at `/`; your working directory decides how relative paths resolve.
- `pwd`, `ls`, and `cd` answer where you are, what is there, and how to move — check them before acting.
- `mkdir`, `cp`, `mv`, `rm` organize the tree; `rm` has no undo, so look before you type.
- `stdin`, `stdout`, and `stderr` are the interfaces that make composition possible; keep results on
  stdout and messages on stderr.
- `>` and `>>` redirect output to a file, `<` reads input from one, and `|` connects one program to the next.
- A repeated pipeline belongs in a script: a hash-bang line plus `chmod +x` makes it a command like any other.
- Permissions record read, write, and execute for owner, group, and others; `x` on a directory means "may enter".
- A Python script reads its arguments from `sys.argv`, and returns an exit status the shell can act on.
- `pathlib`, `os`, and `shutil` express the same operations inside a program.

Further reading: [Unix File System and Command Line Interface](https://molnarai.github.io/DataScienceProgramming/blog/unix-file-system-command-line/).